# Twacha — Acne Severity Model (v2, corrected)

Changes vs. v1: robust dataset loading (folder → grade, no overlapping folders), duplicate / near-duplicate
grouping so test images can't leak into train, single imbalance correction, macro-F1 checkpointing,
frozen-BatchNorm fix, matching val/test preprocessing, ONNX export that works on current PyTorch
(single self-contained file + parity check), and a corrected inference snippet.

**Runtime → GPU recommended (Runtime → Change runtime type → T4).**

**Honest limits:** ~1.4k images, several photos per patient (no patient ID available → near-duplicate grouping is a
partial fix only), no clear-skin class (lowest score ≈ 1.5/10). Report macro-F1, not accuracy.

In [ ]:
!pip install -q huggingface_hub scikit-learn matplotlib seaborn onnx onnxscript onnxruntime pillow

In [ ]:
import os, re, glob, hashlib, random, warnings
from collections import Counter, defaultdict
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Download and inspect the raw repo (look at this output before trusting anything below)

In [ ]:
from huggingface_hub import snapshot_download
DATA_ROOT = snapshot_download(repo_id='ManuelHettich/acne04', repo_type='dataset')
print('Downloaded to:', DATA_ROOT)

EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
for d in sorted(os.listdir(DATA_ROOT)):
    p = os.path.join(DATA_ROOT, d)
    if os.path.isdir(p):
        n = sum(1 for _, _, fs in os.walk(p) for f in fs if f.lower().endswith(EXTS))
        print(f'{d:28s} {n:5d} images')

## 2. Build the grade-labelled file list
Only folders named `acne0_1024 … acne3_1024` are used. Other folders (`all_1024`, `small_*`, `examples`) are
ignored because they overlap with these. **Assumption to verify with the sample grid below:** `acne0` = mildest,
`acne3` = most severe (ACNE04 has no clear-skin class).

In [ ]:
FOLDER_RE = re.compile(r'^acne([0-3])_1024$')
records = []                     # (path, grade)
for d in sorted(os.listdir(DATA_ROOT)):
    m = FOLDER_RE.match(d)
    if not m: 
        continue
    g = int(m.group(1))
    for p in glob.glob(os.path.join(DATA_ROOT, d, '**', '*'), recursive=True):
        if p.lower().endswith(EXTS):
            records.append((p, g))
assert len({g for _, g in records}) == 4, f'Expected 4 grade folders, found: {Counter(g for _, g in records)}'

# exact duplicates (same bytes) -> keep one
seen, uniq = {}, []
for p, g in records:
    h = hashlib.md5(open(p, 'rb').read()).hexdigest()
    if h in seen:
        continue
    seen[h] = g; uniq.append((p, g))
print(f'{len(records)} files -> {len(uniq)} after exact-duplicate removal')
records = uniq
print('Grade distribution:', dict(sorted(Counter(g for _, g in records).items())))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for g in range(4):
    p = next(p for p, gg in records if gg == g)
    axes[g].imshow(Image.open(p).convert('RGB')); axes[g].set_title(f'Grade {g}'); axes[g].axis('off')
plt.suptitle('CHECK: grades should look progressively more severe'); plt.tight_layout(); plt.show()

## 3. Leak-resistant split
Photos of the same person/crops of the same photo look alike. We cluster near-duplicates with a difference hash
(dHash) and keep each cluster entirely inside one split. This is **not** a true patient-wise split (ACNE04 gives
no patient IDs), so treat test scores as slightly optimistic.

In [ ]:
def dhash(path, size=8):
    im = Image.open(path).convert('L').resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int(''.join('1' if b else '0' for b in bits), 2)

hashes = [dhash(p) for p, _ in records]
n = len(records)
parent = list(range(n))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
HAM_T = 5
for i in range(n):
    for j in range(i + 1, n):
        if bin(hashes[i] ^ hashes[j]).count('1') <= HAM_T:
            parent[find(i)] = find(j)
groups = np.array([find(i) for i in range(n)])
y = np.array([g for _, g in records])
print(f'{n} images in {len(set(groups))} near-duplicate groups')

idx = np.arange(n)
sgkf_a = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(sgkf_a.split(idx, y, groups))
sgkf_b = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(sgkf_b.split(trainval_idx, y[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[va_rel]

assert not (set(groups[train_idx]) & set(groups[test_idx])), 'group leak train/test'
assert not (set(groups[train_idx]) & set(groups[val_idx])), 'group leak train/val'
assert not (set(groups[val_idx]) & set(groups[test_idx])), 'group leak val/test'
for name, ix in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:5s} {len(ix):5d}', {int(k): v for k, v in sorted(Counter(y[ix]).items())})
num_classes = 4

## 4. Data pipeline (val/test use the same resize-256 + center-crop-224 geometry as training crops)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class AcneDS(Dataset):
    def __init__(self, recs, tf): self.recs, self.tf = recs, tf
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        p, g = self.recs[i]
        return self.tf(Image.open(p).convert('RGB')), g

mk = lambda ix, tf: AcneDS([records[i] for i in ix], tf)
BATCH_SIZE = 32
train_loader = DataLoader(mk(train_idx, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(mk(val_idx, eval_tf),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(mk(test_idx, eval_tf),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ONE imbalance correction only: class-weighted loss (no weighted sampler on top)
cnt = Counter(y[train_idx]); tot = len(train_idx)
class_w = torch.tensor([tot / (num_classes * cnt[c]) for c in range(num_classes)], dtype=torch.float32)
print('class weights:', class_w.tolist())

## 5. Model (frozen early layers kept in eval mode so BatchNorm statistics don't drift)

In [ ]:
def create_model(num_classes):
    m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    for p in m.features[:14].parameters():
        p.requires_grad = False
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.last_channel, 256), nn.ReLU(),
                                 nn.Dropout(0.2), nn.Linear(256, num_classes))
    return m.to(device)

def set_train_mode(m):
    m.train()
    for mod in m.features[:14].modules():       # frozen part: keep BN stats fixed
        if isinstance(mod, nn.BatchNorm2d):
            mod.eval()

model = create_model(num_classes)
tp = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params {tp:,}, trainable {tr:,} ({tr/tp*100:.1f}%)')
criterion = nn.CrossEntropyLoss(weight=class_w.to(device))
NUM_EPOCHS = 25
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## 6. Train — best checkpoint chosen by **validation macro-F1**

In [ ]:
def run_eval(loader):
    model.eval(); loss_sum, ys, ps = 0.0, [], []
    with torch.no_grad():
        for x, t in loader:
            x, t = x.to(device), t.to(device)
            o = model(x); loss_sum += criterion(o, t).item()
            ys += t.cpu().tolist(); ps += o.argmax(1).cpu().tolist()
    return loss_sum / len(loader), f1_score(ys, ps, average='macro'), float(np.mean(np.array(ys) == np.array(ps)))

best_f1 = -1.0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}
for epoch in range(NUM_EPOCHS):
    set_train_mode(model); run = 0.0
    for x, t in train_loader:
        x, t = x.to(device), t.to(device)
        optimizer.zero_grad(); loss = criterion(model(x), t); loss.backward(); optimizer.step()
        run += loss.item()
    scheduler.step()
    vl, vf1, vacc = run_eval(val_loader)
    history['train_loss'].append(run / len(train_loader)); history['val_loss'].append(vl)
    history['val_f1'].append(vf1); history['val_acc'].append(vacc)
    if vf1 > best_f1:
        best_f1 = vf1; torch.save(model.state_dict(), 'best_acne_model.pth'); flag = '  <- saved'
    else:
        flag = ''
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  train_loss {history["train_loss"][-1]:.3f}  val_loss {vl:.3f}  val_macroF1 {vf1:.3f}  val_acc {vacc:.3f}{flag}')

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
a.plot(history['train_loss'], label='train'); a.plot(history['val_loss'], label='val'); a.set_title('Loss'); a.legend(); a.grid(alpha=.3)
b.plot(history['val_f1'], label='val macro-F1'); b.plot(history['val_acc'], label='val acc'); b.set_title('Validation'); b.legend(); b.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Test-set evaluation (run once, at the end)

In [ ]:
model.load_state_dict(torch.load('best_acne_model.pth', map_location=device)); model.eval()
ys, ps = [], []
with torch.no_grad():
    for x, t in test_loader:
        ps += model(x.to(device)).argmax(1).cpu().tolist(); ys += t.tolist()
names = [f'Grade {i}' for i in range(num_classes)]
print(classification_report(ys, ps, target_names=names, labels=list(range(num_classes)), zero_division=0))
print('Macro-F1:', round(f1_score(ys, ps, average='macro'), 3))
off_by_one = np.mean(np.abs(np.array(ys) - np.array(ps)) <= 1)
print(f'Within +/-1 grade: {off_by_one:.3f}   (the number that matters for a 0-10 score)')
cm = confusion_matrix(ys, ps, labels=list(range(num_classes)))
plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion matrix (test)'); plt.tight_layout(); plt.show()

## 8. Grade → 0–10 score (expected value over class probabilities)

In [ ]:
ANCHORS = [1.5, 4.0, 7.0, 9.5]
def grade_to_score(logits):
    probs = torch.softmax(logits, dim=-1)
    a = torch.tensor(ANCHORS, device=logits.device, dtype=probs.dtype)
    return (probs * a).sum(-1).clamp(0, 10)

x, t = next(iter(test_loader))
with torch.no_grad(): s = grade_to_score(model(x[:6].to(device)))
for i in range(len(s)): print(f'true grade {t[i].item()} -> acne score {s[i].item():.1f}/10')

## 9. Export — one self-contained `.onnx` file, verified against PyTorch

In [ ]:
import onnx, onnxruntime as ort
model.eval().cpu()
dummy = torch.randn(1, 3, 224, 224)
try:
    torch.onnx.export(model, dummy, 'twacha_acne_model.onnx', input_names=['image'], output_names=['logits'],
                      dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17, dynamo=False)
except TypeError:                                   # older torch has no dynamo= argument (and already uses the legacy exporter)
    torch.onnx.export(model, dummy, 'twacha_acne_model.onnx', input_names=['image'], output_names=['logits'],
                      dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)

# make sure weights are INSIDE the .onnx (new exporters may write a sidecar .data file)
m = onnx.load('twacha_acne_model.onnx', load_external_data=True)
onnx.save_model(m, 'twacha_acne_model.onnx', save_as_external_data=False)
for f in glob.glob('twacha_acne_model.onnx.data'): os.remove(f)
onnx.checker.check_model('twacha_acne_model.onnx')

# parity test: PyTorch vs onnxruntime, using a real test batch
sess = ort.InferenceSession('twacha_acne_model.onnx', providers=['CPUExecutionProvider'])
xb = x[:4].cpu()
with torch.no_grad(): ref = model(xb).numpy()
got = sess.run(None, {'image': xb.numpy().astype(np.float32)})[0]
print('max abs diff PyTorch vs ONNX:', float(np.abs(ref - got).max()))
assert np.abs(ref - got).max() < 1e-3, 'ONNX output does not match PyTorch'

torch.save({'model_state_dict': model.state_dict(), 'num_classes': num_classes, 'input_size': 224,
            'normalize_mean': MEAN, 'normalize_std': STD, 'grade_anchors': ANCHORS}, 'twacha_acne_model.pth')
print(f"ONNX size: {os.path.getsize('twacha_acne_model.onnx')/1e6:.1f} MB (should be ~10 MB; if it is under 1 MB the weights are missing)")

In [ ]:
try:
    from google.colab import files
    files.download('twacha_acne_model.onnx'); files.download('twacha_acne_model.pth')
except ImportError:
    print('Not running in Colab - files are in the working directory.')

## 10. Backend inference snippet for `mock_cv.py` (replace the v1 snippet — it crashed on float64 and skipped BGR→RGB)

Preprocessing here matches validation: resize shorter side to 256, center-crop 224, RGB, ImageNet-normalise, float32.

```python
import cv2, numpy as np, onnxruntime as ort

_SESS = ort.InferenceSession("twacha_acne_model.onnx", providers=["CPUExecutionProvider"])
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_ANCHORS = np.array([1.5, 4.0, 7.0, 9.5], dtype=np.float32)

def _preprocess(bgr: np.ndarray) -> np.ndarray:
    h, w = bgr.shape[:2]
    s = 256 / min(h, w)
    bgr = cv2.resize(bgr, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    h, w = bgr.shape[:2]
    top, left = (h - 224) // 2, (w - 224) // 2
    bgr = bgr[top:top + 224, left:left + 224]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - _MEAN) / _STD
    return np.ascontiguousarray(rgb.transpose(2, 0, 1)[None]).astype(np.float32)

def acne_score(bgr: np.ndarray) -> float:
    logits = _SESS.run(None, {"image": _preprocess(bgr)})[0][0]
    p = np.exp(logits - logits.max()); p /= p.sum()
    return float(np.clip((p * _ANCHORS).sum(), 0, 10))
```